In [7]:
import tensorflow as tf
from tensorflow.keras import datasets, layers, models
from tensorflow.keras.optimizers import Adam
import keras
from keras.models import Sequential, Model
from keras.layers import *
from keras.utils import Sequence
from keras.layers import Conv2D, MaxPooling2D
from qkeras import *

from keras.utils import Sequence
from keras.callbacks import CSVLogger
from keras.callbacks import EarlyStopping

import os
import random
from datetime import datetime
#import datetime
import time

import matplotlib.pyplot as plt

import pandas as pd

pi = 3.14159265359

maxval=1e9
minval=1e-9

from tqdm import tqdm
import seaborn as sns

In [8]:
import importlib
import OptimizedDataGenerator_v2

importlib.reload(OptimizedDataGenerator_v2)
from OptimizedDataGenerator_v2 import OptimizedDataGenerator


In [3]:
!ls

3x3_grid_search_train_model_Better_DG-Copy1.ipynb
docs
evaluate.py
from_weights.ipynb
grid_search_train_model_Better_DG.ipynb
loss.py
meanpredicteduncertainty.ipynb
Merge_plan_lab.ipynb
mergeplan.py
mkdocs.yml
model_batchnorm
models.py
myenv.yml
OptimizedDataGeneratorNew.py
OptimizedDataGenerator.py
OptimizedDataGenerator_v2.py
plotting
preselection_processor.py
__pycache__
requirements.txt
site
test_2x2_2x2.csv
test_2x2_3x3.csv
test_2x2_4x4.csv
test_2x2_5x5.csv
test_2x2_6x6.csv
test_2x2_7x7.csv
test_3x3_2x2.csv
test_3x3_3x3.csv
test_3x3_4x4.csv
test_3x3_5x5.csv
test_3x3_6x6.csv
test_3x3_7x7.csv
test_3x3.csv
test_noise_3x3.csv
test_run.py
trained_models
train_model_ben_new.ipynb
train_model_Better_DG.ipynb
train_model_new_example.ipynb
train_model_optimized.ipynb
utils.py


In [9]:
#from dataprep import *
# from OptimizedDataGenerator_v2 import OptimizedDataGenerator
from loss import *
from models import *

In [10]:
dataset_base_dir = "/uscms/home/bweiss/nobackup/smart-pixels/" #/uscms/home/bweiss/nobackup/smart-pixels/dataset_3sr_16x16_50x12P5_parquets/meanfilt2
tfrecords_base_dir = "/uscms/home/acauper/nobackup/SmartPixels/"
tfrecords_base_dir = os.path.join(tfrecords_base_dir, "tfrecords")

dataset_dir_train = os.path.join(dataset_base_dir, "dataset_3sr_16x16_50x12P5_parquets/", 'train')
#dataset_dir_train = os.path.join(dataset_base_dir, "dataset_3sr_16x16_50x12P5_parquets/meanfilt2/", 'train') #different mean filter
#dataset_dir_train = os.path.join(dataset_base_dir, "dataset_3sr_16x16_50x12P5_parquets/meanfilt3", 'train')
dataset_dir_val = os.path.join(dataset_base_dir, "dataset_3sr_16x16_50x12P5_parquets/", 'test')
#dataset_dir_val = os.path.join(dataset_base_dir, "dataset_3sr_16x16_50x12P5_parquets/meanfilt2/", 'test') #different mean filter
#dataset_dir_val = os.path.join(dataset_base_dir, "dataset_3sr_16x16_50x12P5_parquets/meanfilt3", 'test')
tfrecords_dir_train = os.path.join(tfrecords_base_dir, "TFR_train",'no_noise_3sr_16x16')                      #change to no noise
#tfrecords_dir_train = os.path.join(tfrecords_base_dir, "TFR_train",'2x2_3sr_16x16')
#tfrecords_dir_train = os.path.join(tfrecords_base_dir, "TFR_train",'3x3_3sr_16x16') #different mean filter
tfrecords_dir_val = os.path.join(tfrecords_base_dir, "TFR_val",'no_noise_3sr_16x16')                          #change to no noise
#tfrecords_dir_val = os.path.join(tfrecords_base_dir, "TFR_val",'2x2_3sr_16x16')
#tfrecords_dir_val = os.path.join(tfrecords_base_dir, "TFR_val",'3x3_3sr_16x16') #different mean filter

batch_size = 5000
val_batch_size = 5000
train_file_size = 80
val_file_size = 20
#---------------------------

start_time = time.time()
validation_generator = OptimizedDataGenerator(
    dataset_base_dir = dataset_dir_val,
    file_type = "parquet",
    data_format = "3D",
    batch_size = val_batch_size,
    # optimize_batch_size = True,
    file_count = val_file_size,
    to_standardize= True,
    labels_list = ['x-midplane','y-midplane','cotAlpha','cotBeta'],
    input_shape = (2,16,16), # (20,16,16),
    transpose = (0,2,3,1),
    shuffle = False, 
    files_from_end=True,
    select_contained=True,
    noise = -1,                               #NOISE
    tfrecords_dir = tfrecords_dir_val,
    use_time_stamps = [0,19],
    max_workers = 2
)

print("--- Validation generator %s seconds ---" % (time.time() - start_time))

# training generator
start_time = time.time()
training_generator = OptimizedDataGenerator(
    dataset_base_dir = dataset_dir_train,
    file_type = "parquet",
    data_format = "3D",
    batch_size = batch_size,
    # optimize_batch_size = True,
    file_count = train_file_size,
    to_standardize= True,
    labels_list = ['x-midplane','y-midplane','cotAlpha','cotBeta'],
    input_shape = (2,16,16), # (20,16,16),
    transpose = (0,2,3,1),
    shuffle = False, # True 
    select_contained=True,
    noise = -1,
    tfrecords_dir = tfrecords_dir_train,
    use_time_stamps = [0,19],
    max_workers = 2
)
print("--- Training generator %s seconds ---" % (time.time() - start_time))

Processing Files...: 100%|██████████| 20/20 [00:18<00:00,  1.07it/s]


Directory /uscms/home/acauper/nobackup/SmartPixels/tfrecords/TFR_val/no_noise_3sr_16x16 does not exist and cannot be removed.


Saving batches as TFRecords: 100%|██████████| 21/21 [00:22<00:00,  1.08s/it]


Metadata saved successfully ast /uscms/home/acauper/nobackup/SmartPixels/tfrecords/TFR_val/no_noise_3sr_16x16/metadata.json
Loading metadata from /uscms/home/acauper/nobackup/SmartPixels/tfrecords/TFR_val/no_noise_3sr_16x16/metadata.json


--- Validation generator 47.83184552192688 seconds ---


Processing Files...: 100%|██████████| 80/80 [00:50<00:00,  1.60it/s]


Directory /uscms/home/acauper/nobackup/SmartPixels/tfrecords/TFR_train/no_noise_3sr_16x16 does not exist and cannot be removed.


Saving batches as TFRecords: 100%|██████████| 84/84 [02:27<00:00,  1.76s/it]


Metadata saved successfully ast /uscms/home/acauper/nobackup/SmartPixels/tfrecords/TFR_train/no_noise_3sr_16x16/metadata.json
Loading metadata from /uscms/home/acauper/nobackup/SmartPixels/tfrecords/TFR_train/no_noise_3sr_16x16/metadata.json


--- Training generator 204.77345991134644 seconds ---


In [ ]:
base_dir = f'./trained_models/model-3x3_5x5-checkpoints'
checkpoint_files = [os.path.join(base_dir, f) for f in os.listdir(base_dir) if f.endswith('.hdf5')]
latest_checkpoint = max(checkpoint_files, key=os.path.getmtime)

print(f"Loading model from {latest_checkpoint}")
model.load_weights(latest_checkpoint)
print(model.load_weights(latest_checkpoint))

In [ ]:
#list = [2, 3, 4, 5, 6, 7]
list = [3]
for i, kern in enumerate(list):
    model=CreateModel((16,16,2),n_filters=5,pool_size=3,conv_kernel_size=(kern,kern)) #had to add conv_kernel_size
    model.compile(
        optimizer=tf.keras.optimizers.Nadam(learning_rate=1e-3),
        loss=custom_loss
    )
    fingerprint = '%08x' % random.randrange(16**8)
    timestamp = datetime.now().strftime('%Y%m%d-%H%M%S')
    os.makedirs("trained_models", exist_ok=True)
    #base_dir = f'./trained_models/model-2x2_{kern}x{kern}-{fingerprint}-checkpoints'   #change mean filter title name
    #base_dir = f'./trained_models/model-3x3_{kern}x{kern}-{fingerprint}-checkpoints'    #change mean filter title name
    base_dir = f'./trained_models/model_no_noise-{kern}x{kern}-{fingerprint}-checkpoints'   #change to no noise
    os.makedirs(base_dir, exist_ok=True)  
    checkpoint_filepath = base_dir + '/weights.{epoch:03d}-t{loss:.2f}-v{val_loss:.2f}.hdf5'
    #checkpoint_filepath = base_dir + '/weights.'+str(epoch).zfill(3)+'-t{loss:.2f}-v{val_loss:.2f}.hdf5'
#model.summary()

    from tensorflow.keras.callbacks import CSVLogger, EarlyStopping, ModelCheckpoint, Callback
    
    early_stopping_patience = 50
    
    class CustomModelCheckpoint(ModelCheckpoint):
        def on_epoch_end(self, epoch, logs=None):
            super().on_epoch_end(epoch, logs)
            checkpoints = [f for f in os.listdir(base_dir) if f.startswith('weights')]
            if len(checkpoints) > 1:
                checkpoints.sort()
                for checkpoint in checkpoints[:-1]:
                    os.remove(os.path.join(base_dir, checkpoint))
    
    es = EarlyStopping(patience=early_stopping_patience, restore_best_weights=True)
    
    mcp = CustomModelCheckpoint(
        filepath=checkpoint_filepath,
        save_weights_only=True,
        monitor='val_loss',
        save_best_only=True,
        save_freq='epoch',
        verbose=1
    )
    
    csv_logger = CSVLogger(f'{base_dir}/training_log.csv', append=True)

    history = model.fit(
            x=training_generator,
            validation_data=validation_generator,
            callbacks=[es, mcp, csv_logger],
            epochs=600,
            shuffle=False,
            verbose=1
        )

    checkpoint_files = [os.path.join(base_dir, f) for f in os.listdir(base_dir) if f.endswith('.hdf5')]      #
    latest_checkpoint = max(checkpoint_files, key=os.path.getmtime)                                          #
    print(f"Loading model from {latest_checkpoint}")                                                         #
    model.load_weights(latest_checkpoint)                                                                    #
    training_history = pd.read_csv(f'{base_dir}/training_log.csv')
    plt.scatter(training_history.index, training_history['loss'])
    plt.scatter(training_history.index, training_history['val_loss'])
    plt.legend(['training', 'validation'])
    plt.grid(True)
    plt.xlabel('Epochs')
    plt.ylabel('NLL loss')
    plt.title(f"{kern}x{kern} Loss wrt Epochs")
    plt.tight_layout() 
    
    plt.savefig(os.path.join(base_dir,'training_hist.png'))
    plt.show()
    plt.close()#######################################################

    y_scaled_true = model.predict(validation_generator)

    complete_truth = None
    for _, y in tqdm(validation_generator):
            if complete_truth is None:
                complete_truth = y
            else:
                complete_truth = np.concatenate((complete_truth, y), axis=0)

    df = pd.DataFrame(y_scaled_true,columns=['x','M11','y','M22','cotA','M33','cotB','M44','M21','M31','M32','M41','M42','M43']) #scaled
    
    df['xtrue'] = complete_truth[:,0]
    df['ytrue'] = complete_truth[:,1]
    df['cotAtrue'] = complete_truth[:,2]
    df['cotBtrue'] = complete_truth[:,3]
    df['M11'] = minval+tf.math.maximum(df['M11'], 0)
    df['M22'] = minval+tf.math.maximum(df['M22'], 0)
    df['M33'] = minval+tf.math.maximum(df['M33'], 0)
    df['M44'] = minval+tf.math.maximum(df['M44'], 0)
    
    df['sigmax'] = abs(df['M11'])
    df['sigmay'] = np.sqrt(df['M21']**2 + df['M22']**2)
    df['sigmacotA'] = np.sqrt(df['M31']**2+df['M32']**2+df['M33']**2)
    df['sigmacotB'] = np.sqrt(df['M41']**2+df['M42']**2+df['M43']**2+df['M44']**2)
    
    residuals = df['xtrue'] - df['x']
    residualsy = df['ytrue'] - df['y']
    residualsA = df['cotAtrue'] - df['cotA']
    residualsB = df['cotBtrue'] - df['cotB']
    
    #df.to_csv(f"test_2x2_{kern}x{kern}.csv",header=True,index=False) #mean filter
    #df.to_csv(f"test_3x3_{kern}x{kern}.csv",header=True,index=False)
    df.to_csv(f"test_no_noise_{kern}x{kern}.csv",header=True,index=False)               #change to no noise

    def residual_plot(ax, thisdf, var1, var2, name, title, scaling=1.0):
        
        nbins = 15
        
        var1_scaled = thisdf[var1] * scaling
        var2_scaled = thisdf[var2] * scaling
        residual_scaled = var1_scaled - var2_scaled
        
        xmin = np.min(var1_scaled)
        xmax = np.max(var1_scaled)
        
        step = 1.0*(xmax-xmin)/nbins
        
        x = sns.regplot(x=var1_scaled, y=residual_scaled, x_bins=np.linspace(xmin,xmax,nbins), fit_reg=None, marker='.', ax=ax)
        ax.set_xlabel('True ' + name)
        ax.set_ylabel('True - predicted ' + name)
        ax.set_title(title)
        
        thisdf['residual'+var2] = residual_scaled
        print(var1)
        
        means = []
        upbar = []
        downbar = []
        for i in range(0,nbins):
            means += [np.mean(thisdf['residual'+var2][(var1_scaled>xmin + i*step) & (var1_scaled<xmin + (i+1)*step)])]
            upbar += [means[i] + np.mean(thisdf['sigma'+var2][(var1_scaled>xmin + i*step) & (var1_scaled<xmin + (i+1)*step)] * scaling)]
            downbar += [means[i] - np.mean(thisdf['sigma'+var2][(var1_scaled>xmin + i*step) & (var1_scaled<xmin + (i+1)*step)] * scaling)]
        ax.fill_between(x=np.linspace(xmin,xmax,nbins),y1=upbar,y2=downbar, alpha=0.2)
    
    def inverse_cot(cota):
        a = np.arctan(1.0/cota)
        a[np.where(a<0)] = a[np.where(a<0)] + pi
        return a    
    
    def residual_plot_deg(ax, thisdf, var1, var2, name, title, scaling=1.0):
        # positions
        if 'cot' not in var1:
            residual_plot(ax, thisdf, var1, var2, name, scaling=scaling)
            return
    
        thisdf['angle'] = inverse_cot(thisdf[var2].values * scaling)*180/pi
        
        thisdf['angleup'] = abs(inverse_cot((thisdf[var2].values + thisdf['sigma'+var2].values) * scaling)*180/pi - thisdf['angle'])
        thisdf['angledown'] = abs(inverse_cot((thisdf[var2].values - thisdf['sigma'+var2].values) * scaling)*180/pi - thisdf['angle'])
        thisdf['angletrue'] = inverse_cot(thisdf[var1].values * scaling)*180/pi
            
        var1 = 'angletrue'
        var2 = 'angle'
        
        nbins = 15
        xmin = np.min(thisdf[var1])
        xmax = np.max(thisdf[var1])
        
        step = 1.0*(xmax-xmin)/nbins
            
        x = sns.regplot(x=thisdf[var1], y=(thisdf[var1]-thisdf[var2]), x_bins=np.linspace(xmin,xmax,nbins), fit_reg=None, marker='.', ax=ax)
        ax.set_xlabel('True ' + name)
        ax.set_ylabel('True - predicted ' + name)
        ax.set_title(title)
        
        thisdf['residual'+var2] = (thisdf[var1]-thisdf[var2])
        print(var1)
        
        means = []    
        upbar = []
        downbar = []
        for i in range(0,nbins):
            means += [np.mean(thisdf['residual'+var2][(thisdf[var1]>xmin + i*step) & (thisdf[var1]<xmin + (i+1)*step)])]
            upbar += [means[i] + np.mean(thisdf['angleup'][(thisdf[var1]>xmin + i*step) & (thisdf[var1]<xmin + (i+1)*step)])]
            downbar += [means[i] - np.mean(thisdf['angledown'][(thisdf[var1]>xmin + i*step) & (thisdf[var1]<xmin + (i+1)*step)])]
        #ax.scatter(x=np.linspace(xmin,xmax,nbins),y=means)
        ax.fill_between(x=np.linspace(xmin,xmax,nbins),y1=upbar,y2=downbar, alpha=0.2)

    fig, axes = plt.subplots(2,2,figsize=(8,6))
    fig.tight_layout(pad=4.0)
    #residual_plot(axes[0][0],df,'xtrue','x',r'$x$ [um]', title=f"{kern}x{kern} x-mid Residual Noise", scaling=75.0)   #different for mean filter removed 2x2,  from all
    residual_plot(axes[0][0],df,'xtrue','x',r'$x$ [um]', title=f"{kern}x{kern} x-mid Residual No Noise", scaling=75.0)   #different for mean filter removed 3x3, from all
    axes[0][0].plot([-25,-25],[-10,10],color='gray',linestyle=':')
    axes[0][0].plot([25,25],[-10,10],color='gray',linestyle=':')
    #residual_plot(axes[0][1],df,'ytrue','y',r'$y$ [um]', title=f"{kern}x{kern} y-mid Residual Noise", scaling=18.75)   #different for mean filter
    residual_plot(axes[0][1],df,'ytrue','y',r'$y$ [um]', title=f"{kern}x{kern} y-mid Residual No Noise", scaling=18.75)   #different for mean filter
    axes[0][1].plot([-6.25,-6.25],[-2,2],color='gray',linestyle=':')
    axes[0][1].plot([6.25,6.25],[-2,2],color='gray',linestyle=':')
    #residual_plot_deg(axes[1][0],df,'cotAtrue','cotA',r'$\alpha$ [deg]', title=f"{kern}x{kern} cotA Residual Noise", scaling=8.0)   #different for mean filter
    residual_plot_deg(axes[1][0],df,'cotAtrue','cotA',r'$\alpha$ [deg]', title=f"{kern}x{kern} cotA Residual No Noise", scaling=8.0)   #different for mean filter
    axes[1][0].plot([90,90],[-10,10],color='gray',linestyle=':')
    #residual_plot_deg(axes[1][1],df,'cotBtrue','cotB',r'$\beta$ [deg]', title=f"{kern}x{kern} cotB Residual Noise", scaling=0.5)   #different for mean filter
    residual_plot_deg(axes[1][1],df,'cotBtrue','cotB',r'$\beta$ [deg]', title=f"{kern}x{kern} cotB Residual No Noise", scaling=0.5)   #different for mean filter
    axes[1][1].plot([90,90],[-10,10],color='gray',linestyle=':')

    save_fig_path = os.path.join(base_dir, 'summary.png')
    plt.savefig(save_fig_path)
    plt.close()################################################
    if kern == 3:
        break

Epoch 1/600
84/84 [==============================] - ETA: 0s - loss: 37939.3711
Epoch 1: val_loss improved from inf to 13396.29102, saving model to ./trained_models/model_no_noise-3x3-1aaa3239-checkpoints/weights.001-t37939.37-v13396.29.hdf5
84/84 [==============================] - 37s 363ms/step - loss: 37939.3711 - val_loss: 13396.2910
Epoch 2/600
84/84 [==============================] - ETA: 0s - loss: 11159.7295
Epoch 2: val_loss improved from 13396.29102 to 8301.18945, saving model to ./trained_models/model_no_noise-3x3-1aaa3239-checkpoints/weights.002-t11159.73-v8301.19.hdf5
84/84 [==============================] - 19s 217ms/step - loss: 11159.7295 - val_loss: 8301.1895
Epoch 3/600
15/84 [====>.........................] - ETA: 8s - loss: 7963.2417

In [8]:
base_dir = f'./trained_models/model-2x2_*-checkpoints'

In [ ]:
base_dir = f'./trained_models/model-2x2_{kern}x{kern}-{fingerprint}-checkpoints'
checkpoint_files = [os.path.join(base_dir, f) for f in os.listdir(base_dir) if f.endswith('.hdf5')]
latest_checkpoint = max(checkpoint_files, key=os.path.getmtime)
training_cp_path = os.path.join(base_dir, 'training_log.csv')
training_history = pd.read_csv(training_cp_path)

In [ ]:
fig, axes = plt.subplots(2,2,figsize=(8,6))
fig.tight_layout(pad=4.0)
residual_plot(axes[0][0],df,'xtrue','x',r'$x$ [um]', scaling=75.0)
axes[0][0].plot([-25,-25],[-10,10],color='gray',linestyle=':')
axes[0][0].plot([25,25],[-10,10],color='gray',linestyle=':')
residual_plot(axes[0][1],df,'ytrue','y',r'$y$ [um]', scaling=18.75)
axes[0][1].plot([-6.25,-6.25],[-2,2],color='gray',linestyle=':')
axes[0][1].plot([6.25,6.25],[-2,2],color='gray',linestyle=':')
residual_plot_deg(axes[1][0],df,'cotAtrue','cotA',r'$\alpha$ [deg]', scaling=8.0)
axes[1][0].plot([90,90],[-10,10],color='gray',linestyle=':')
residual_plot_deg(axes[1][1],df,'cotBtrue','cotB',r'$\beta$ [deg]', scaling=0.5)
axes[1][1].plot([90,90],[-10,10],color='gray',linestyle=':')

save_fig_path = os.path.join(base_dir, '2x2_total_summary.png')
plt.savefig(save_fig_path)

In [6]:
(base_dir)

['/',
 't',
 'r',
 'a',
 'i',
 'n',
 'e',
 'd',
 '_',
 'm',
 'o',
 'd',
 'e',
 'l',
 's',
 '/',
 'm',
 'o',
 'd',
 'e',
 'l',
 '-',
 '2',
 'x',
 '2',
 '_',
 '*',
 '-',
 'c',
 'h',
 'e',
 'c',
 'k',
 'p',
 'o',
 'i',
 'n',
 't',
 's']